# Parser V5 Development - Analyse et amélioration du parsing

Ce notebook sert à :
1. Analyser la structure brute des PDFs FFVB
2. Évaluer la qualité du parser V4
3. Développer et tester le parser V5

In [1]:
import sys
sys.path.insert(0, '/home/vincheetah/Documents/Programmation/Python/PyVolley/src')

import pdfplumber
from pathlib import Path
import json
from pprint import pprint

In [2]:
# Load a sample PDF
pdf_path = Path('/home/vincheetah/Documents/Programmation/Python/PyVolley/data/pdfs/2025-2026/ABCCS/Elite M. Poule A/ABCCS_EMA001.pdf')

with pdfplumber.open(str(pdf_path)) as pdf:
    print(f'Pages: {len(pdf.pages)}')
    page = pdf.pages[0]
    
    # Full text
    full_text = page.extract_text()
    print('=== FULL TEXT ===')
    print(full_text)

Pages: 1
=== FULL TEXT ===
EMA - ELITE MASCULINE - POULE A Match: EMA001 - Jour: 01
Ville: SAINT MARTIN D'HÈRES Samedi 20 Septembre 2025 à 20h30
Salle: CSU - GRAND GYMNASE SENIOR | MASCULIN
Compétitions Nationales SENIORS GRENOBLE V.UNIVERSITE CLUB GPSO ACBB
S GRENOBLE V.UNIVERSITE Début: 20:28 S GPSO ACBB Fin: 20:52 R S GPSO ACBB Début: 20:55 S GRENOBLE V.UNIVERSITE Fin: 21:25 R
Ordre de Service E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 1 1 1 2 E I II III IV V VI 1 2 1 1 1 2 2 2 1 2 I II III IV V VI 1 2 1 1 1 2
Formation de Départ 51 8 11 5 17 9 31323 3 10 24 8 7 17 3 8 7 17 3 10 24 31323 9 51 8 11 5 17 313
Joueur N° T 6 4 5 1 1 4 5 2 2 4 5 9 11 4 4 5 T 5 9 4 4 5 1 1 4 5 2 2 4 5 26 4 5 1 1 4 5
Remplaçants 24:12 616 6:13 6:13 11:23 6 19:10 19:10 22:16 616 4:5 616
Score 717 7 717 717
818 9:18 9:18 12:24 8 23:18 23:18 22:17 818 818
1 5 1 0 1 2 4 8 10 1 9 0 1 2 9 0 X 1 2 3 4 5 1 9 0 2 0 1 3 4 8 10 1 9 0 1 2 9 0 X 1 2 3 4 5 1 9 0 19
2 6 16 17 18 21 23 25 T 6 7 8 9 10 12 T

In [3]:
# Examine all tables
with pdfplumber.open(str(pdf_path)) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()
    
    print(f'Number of tables: {len(tables)}')
    for i, table in enumerate(tables):
        print(f'\n=== TABLE {i} === ({len(table)} rows, max {max(len(r) for r in table if r)} cols)')
        for j, row in enumerate(table):
            print(f'  Row {j}: {row}')

Number of tables: 5

=== TABLE 0 === (44 rows, max 49 cols)
  Row 0: [None, None, None, None, None, None, None, None, None, None, None, 'S\nE\nT\n1', None, 'GRENOBLE V.UNIVERSITE Début: 20:28 S', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'GPSO ACBB Fin: 20:52 R', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
  Row 1: ['Ordre de Service', None, None, None, None, None, None, None, None, None, None, None, None, 'I', None, 'II', None, None, 'III', None, 'IV', None, 'V', None, 'VI', None, '11121\n21222\n31323\n41424\n51525\n616\n717\n818\n919\n1020', None, None, None, None, 'I', None, 'II', None, 'III', None, 'IV', None, 'V', None, 'VI', None, '111\n212\n3\n4\n5\n6\n7\n8\n9\n10', None, None, None, None, None]
  Row 2: ['Formation de Départ', None, None, None, None, None, None, None, None, None, None, None, None, '51', None, '8', None, None, '11', None, '5', None, '17', None, '

In [ ]:
# Examine words with positions
with pdfplumber.open(str(pdf_path)) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()
    
    print(f'Total words: {len(words)}')
    print('\nFirst 50 words with positions:')
    for w in words[:50]:
        print(f"  x0={w['x0']:.1f} top={w['top']:.1f} text='{w['text']}'")

In [4]:
# Test the v4 parser
from pyvolley.parsers.v4 import MatchSheetParserV4

parser = MatchSheetParserV4()
result = parser.parse(pdf_path)

print(f'Success: {result.success}')
print(f'Parse time: {result.parse_time_ms:.1f}ms')
print(f'Fields: {result.fields_extracted}/{result.fields_total} ({result.completeness:.1%})')
print(f'Errors: {result.errors}')
print(f'Warnings: {result.warnings}')

Success: True
Parse time: 564.9ms
Fields: 61/35 (174.3%)
Errors: []
Warnings: []


In [5]:
# Examine parsed match data in detail
m = result.match
if m:
    print('=== MATCH INFO ===')
    print(f'Code: {m.code_match}')
    print(f'Date: {m.date}')
    print(f'Heure: {m.heure}')
    print(f'Lieu: {m.lieu}')
    print(f'Salle: {m.salle}')
    print(f'Ligue: {m.ligue}')
    print(f'Competition: {m.competition}')
    print(f'Journée: {m.journee}')
    print(f'Saison: {m.saison}')
    print(f'Genre: {m.genre}')
    print(f'Catégorie: {m.categorie}')
    print()
    print(f'=== RESULT ===')
    print(f'Vainqueur: {m.vainqueur_nom}')
    print(f'Score final: {m.score_final}')
    print(f'Sets A/B: {m.sets_a}/{m.sets_b}')
    print(f'Durée: {m.duree_totale}')
    print()
    print(f'=== EQUIPE A: {m.equipe_a.nom} ===')
    print(f'Entraineur: {m.equipe_a.entraineur}')
    print(f'Assistant: {m.equipe_a.assistant}')
    print(f'Joueurs ({len(m.equipe_a.joueurs)}):')
    for j in m.equipe_a.joueurs:
        flags = []
        if j.est_capitaine: flags.append('CAP')
        if j.est_libero: flags.append('LIB')
        print(f'  #{j.numero} {j.nom} {j.prenom} (Lic: {j.licence}) {" ".join(flags)}')
    print(f'Libéros ({len(m.equipe_a.liberos)}):')
    for j in m.equipe_a.liberos:
        print(f'  #{j.numero} {j.nom} {j.prenom} (Lic: {j.licence})')
    print()
    print(f'=== EQUIPE B: {m.equipe_b.nom} ===')
    print(f'Entraineur: {m.equipe_b.entraineur}')
    print(f'Assistant: {m.equipe_b.assistant}')
    print(f'Joueurs ({len(m.equipe_b.joueurs)}):')
    for j in m.equipe_b.joueurs:
        flags = []
        if j.est_capitaine: flags.append('CAP')
        if j.est_libero: flags.append('LIB')
        print(f'  #{j.numero} {j.nom} {j.prenom} (Lic: {j.licence}) {" ".join(flags)}')
    print(f'Libéros ({len(m.equipe_b.liberos)}):')
    for j in m.equipe_b.liberos:
        print(f'  #{j.numero} {j.nom} {j.prenom} (Lic: {j.licence})')
    print()
    print(f'=== SETS ({len(m.sets)}) ===')
    for s in m.sets:
        print(f'  Set {s.numero}: {s.score_a}-{s.score_b} (durée: {s.duree_minutes}min, service: {s.service_initial})')
        print(f'    Début: {s.debut} Fin: {s.fin}')
        if s.formation_a:
            print(f'    Formation A: {s.formation_a.as_dict()}')
        if s.formation_b:
            print(f'    Formation B: {s.formation_b.as_dict()}')
        print(f'    Timeouts A: {[(t.score_a, t.score_b) for t in s.timeouts_a]}')
        print(f'    Timeouts B: {[(t.score_a, t.score_b) for t in s.timeouts_b]}')
    print()
    print(f'=== ARBITRES ({len(m.arbitres)}) ===')
    for a in m.arbitres:
        print(f'  {a.role}: {a.nom} {a.prenom} (Lic: {a.licence}, Ligue: {a.ligue})')
    print()
    print(f'=== SANCTIONS ({len(m.sanctions)}) ===')
    for s in m.sanctions:
        print(f'  {s.type} Set {s.set_numero} Equipe {s.equipe} Joueur #{s.joueur_numero} Score {s.score_a}-{s.score_b}')

=== MATCH INFO ===
Code: EMA001
Date: 2025-09-20
Heure: 20:30:00
Lieu: SAINT MARTIN D'HÈRES S
Salle: CSU - GRAND GYMNASE
Ligue: Ligue L
Competition: EMA - ELITE MASCULINE - POULE A
Journée: 01
Saison: 2025-2026
Genre: Genre.MASCULIN
Catégorie: Categorie.SENIOR

=== RESULT ===
Vainqueur: GRENOBLE V.UNIVERSITE
Score final: 3/1
Sets A/B: 3/1
Durée: 2h9

=== EQUIPE A: GRENOBLE V.UNIVERSITE CLUB ===
Entraineur: VAN DEN ESHOF DAMIEN
Assistant: None
Joueurs (9):
  #01 JOUFFREY AUDRIC (Lic: 2319587) LIB
  #05 CHONÉ TANGUY (Lic: 1892259) 
  #06 FEOUGIER CHARLIE (Lic: 2166846) 
  #08 RATAHIRY ELIJAH (Lic: 1895636) 
  #09 GUERTIN PIERRE (Lic: 2147062) 
  #11 VRBAN MIROSLAV (Lic: 2185796) 
  #17 RAJOHARIVELO TOKY (Lic: 1894940) 
  #26 HAMMOUCHE RANI (Lic: 2337157) 
  #51 MORACCHINI CLEMENT (Lic: 1801350) CAP
Libéros (1):
  #01 JOUFFREY AUDRIC (Lic: 2319587)

=== EQUIPE B: GPSO ACBB ===
Entraineur: TANGUY ERWAN
Assistant: None
Joueurs (11):
  #03 THE OWONA IVAN (Lic: 2593181) 
  #04 GROSJEAN VICTOR

In [6]:
# Test with multiple PDFs to see pattern
import glob

pdf_dir = Path('/home/vincheetah/Documents/Programmation/Python/PyVolley/data/pdfs/2025-2026/ABCCS/Elite M. Poule A/')
pdf_files = sorted(pdf_dir.glob('*.pdf'))[:5]

for pdf_file in pdf_files:
    result = parser.parse(pdf_file)
    m = result.match
    if m:
        print(f'{pdf_file.name}: {m.equipe_a.nom} vs {m.equipe_b.nom} -> {m.score_final} | '
              f'Joueurs A: {len(m.equipe_a.joueurs)}, B: {len(m.equipe_b.joueurs)} | '
              f'Sets: {len(m.sets)} | Arbitres: {len(m.arbitres)} | '
              f'Warnings: {len(result.warnings)}')
        if result.warnings:
            for w in result.warnings:
                print(f'  ⚠ {w}')
    else:
        print(f'{pdf_file.name}: FAILED - {result.errors}')

ABCCS_EMA001.pdf: GRENOBLE V.UNIVERSITE CLUB vs GPSO ACBB -> 3/1 | Joueurs A: 9, B: 11 | Sets: 4 | Arbitres: 4 | Warnings: 0
ABCCS_EMA002.pdf: CONFLANS-ANDRESY-JOUY VB vs VOLLEY CLUB HYERES/PIERREFEU L -> 3/1 | Joueurs A: 13, B: 12 | Sets: 4 | Arbitres: 4 | Warnings: 0
ABCCS_EMA003.pdf: VOLLEY-BALL ARLESIEN vs AVIGNON VOLLEY BALL -> 3/1 | Joueurs A: 12, B: 10 | Sets: 4 | Arbitres: 4 | Warnings: 0
ABCCS_EMA004.pdf: AMIENS METROPOLE VOLLEY vs VC BELLAING/PORTE DU HAINAUT -> 3/0 | Joueurs A: 10, B: 11 | Sets: 3 | Arbitres: 4 | Warnings: 0
ABCCS_EMA005.pdf: LOISIRS INTER SPORT ST PIERRE vs VC MICHELET HALLUIN -> 3/2 | Joueurs A: 13, B: 9 | Sets: 5 | Arbitres: 4 | Warnings: 0


# V5 Parser Debugging — Changements, Timeouts, Officiels

In [7]:
# Examine raw table structure for sets — focus on substitution and timeout rows
import pdfplumber, re, sys
sys.path.insert(0, '/home/vincheetah/Documents/Programmation/Python/PyVolley/src')

pdf_path = '/home/vincheetah/Documents/Programmation/Python/PyVolley/data/pdfs/2025-2026/ABCCS/Elite M. Poule A/ABCCS_EMA001.pdf'

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()

# Identify main table (largest)
main_table = max(tables, key=lambda t: len(t))
print(f"Main table: {len(main_table)} rows x {max(len(r) for r in main_table)} cols")

# Find SET sections
set_pat = re.compile(r'S\s*E\s*T\s*(\d)')
for i, row in enumerate(main_table):
    if not row:
        continue
    for cell in row:
        if cell and set_pat.search(str(cell).replace('\n', ' ')):
            print(f"\n=== SET section at row {i} ===")
            # Print rows +0 through +9 (full set section)
            for offset in range(10):
                ri = i + offset
                if ri < len(main_table):
                    r = main_table[ri]
                    # Show only non-None cells with their column index
                    cells = [(j, str(v).strip()) for j, v in enumerate(r) if v and str(v).strip()]
                    print(f"  +{offset} (row {ri}): {cells}")
            break  # Only first set for now

Main table: 44 rows x 49 cols

=== SET section at row 0 ===
  +0 (row 0): [(11, 'S\nE\nT\n1'), (13, 'GRENOBLE V.UNIVERSITE Début: 20:28 S'), (31, 'GPSO ACBB Fin: 20:52 R')]
  +1 (row 1): [(0, 'Ordre de Service'), (13, 'I'), (15, 'II'), (18, 'III'), (20, 'IV'), (22, 'V'), (24, 'VI'), (26, '11121\n21222\n31323\n41424\n51525\n616\n717\n818\n919\n1020'), (31, 'I'), (33, 'II'), (35, 'III'), (37, 'IV'), (39, 'V'), (41, 'VI'), (43, '111\n212\n3\n4\n5\n6\n7\n8\n9\n10')]
  +2 (row 2): [(0, 'Formation de Départ'), (13, '51'), (15, '8'), (18, '11'), (20, '5'), (22, '17'), (24, '9'), (31, '3'), (33, '10'), (35, '24'), (37, '8'), (39, '7'), (41, '17')]
  +3 (row 3): [(0, 'Remplaçants'), (6, 'Joueur N°'), (24, '6'), (31, '9'), (37, '11'), (41, '4')]
  +4 (row 4): [(6, 'Score'), (24, '24:12'), (31, '6:13'), (37, '6:13'), (41, '11:23')]
  +5 (row 5): [(31, '9:18'), (37, '9:18'), (41, '12:24')]
  +6 (row 6): [(0, 'Tours au service'), (6, '1'), (8, '5'), (13, '0'), (15, '1'), (18, '2'), (20, '4'), (22, 

In [8]:
# Examine secondary table (sets 2 and 4) for substitutions/timeouts
secondary_table = sorted(tables, key=lambda t: len(t), reverse=True)[1]
print(f"Secondary table: {len(secondary_table)} rows x {max(len(r) for r in secondary_table)} cols")

for i, row in enumerate(secondary_table):
    if not row:
        continue
    for cell in row:
        if cell and set_pat.search(str(cell).replace('\n', ' ')):
            set_num = set_pat.search(str(cell).replace('\n', ' ')).group(1)
            print(f"\n=== SET {set_num} section at row {i} ===")
            for offset in range(10):
                ri = i + offset
                if ri < len(secondary_table):
                    r = secondary_table[ri]
                    cells = [(j, str(v).strip()) for j, v in enumerate(r) if v and str(v).strip()]
                    print(f"  +{offset} (row {ri}): {cells}")
            break

Secondary table: 20 rows x 27 cols

=== SET 2 section at row 0 ===
  +0 (row 0): [(0, 'S\nE\nT\n2'), (1, 'GPSO ACBB Début: 20:55 S'), (14, 'GRENOBLE V.UNIVERSITE Fin: 21:25 R')]
  +1 (row 1): [(1, 'I'), (3, 'II'), (5, 'III'), (7, 'IV'), (9, 'V'), (11, 'VI'), (13, '11121\n21222\n31323\n41424\n51525\n616\n717\n818\n919\n1020'), (14, 'I'), (16, 'II'), (18, 'III'), (20, 'IV'), (22, 'V'), (24, 'VI'), (26, '111\n212\n313\n414\n515\n616\n717\n818\n919\n10')]
  +2 (row 2): [(1, '8'), (3, '7'), (5, '17'), (7, '3'), (9, '10'), (11, '24'), (14, '9'), (16, '51'), (18, '8'), (20, '11'), (22, '5'), (24, '17')]
  +3 (row 3): [(1, '5'), (7, '9'), (11, '4'), (14, '26')]
  +4 (row 4): [(1, '19:10'), (7, '19:10'), (11, '22:16'), (14, '4:5')]
  +5 (row 5): [(1, '23:18'), (7, '23:18'), (11, '22:17')]
  +6 (row 6): [(1, '0'), (3, '1'), (5, '3'), (7, '4'), (9, '8'), (11, '10'), (14, 'X'), (16, '1'), (18, '2'), (20, '3'), (22, '4'), (24, '5')]
  +7 (row 7): [(1, '13'), (3, '17'), (5, '18'), (7, '20'), (9, '21

In [9]:
# Examine arbitre/sanctions/officials section in main table (rows 30+)
print("=== MAIN TABLE: bottom rows (arbitres/sanctions area) ===")
for i in range(30, len(main_table)):
    row = main_table[i]
    cells = [(j, str(v).strip()) for j, v in enumerate(row) if v and str(v).strip()]
    if cells:
        print(f"  Row {i}: {cells}")

# Also check the players table for official info
print("\n=== PLAYERS TABLE ===")
players_table = sorted(tables, key=lambda t: len(t))[1]  # smallest meaningful table
print(f"Players table: {len(players_table)} rows x {max(len(r) for r in players_table if r)} cols")
for i, row in enumerate(players_table):
    cells = [(j, str(v).strip()) for j, v in enumerate(row) if v and str(v).strip()]
    if cells:
        print(f"  Row {i}: {cells}")

=== MAIN TABLE: bottom rows (arbitres/sanctions area) ===
  Row 30: [(0, 'SANCTIONS'), (6, 'DEMANDE NON FONDEE'), (13, 'REMARQUES')]
  Row 31: [(6, 'EQU.A EQU.B')]
  Row 32: [(0, 'A'), (1, 'P'), (3, 'E'), (5, 'D'), (6, 'A/B'), (8, 'Set'), (9, 'Score')]
  Row 34: [(13, 'APPROBATION')]
  Row 35: [(13, 'Arbitres'), (16, 'NOM Prénom'), (26, 'Ligue'), (30, 'Licence'), (33, 'Signature')]
  Row 36: [(13, '1er'), (16, 'CHOINARD MICHAEL'), (26, 'PAC'), (30, '1458903')]
  Row 37: [(13, '2ème'), (16, 'HUMBERT RIDET DYLAN'), (26, 'PAC'), (30, '1762370')]
  Row 38: [(13, 'Marqueur'), (16, 'AZAR NICOLAS'), (26, 'ARA'), (30, '2682444')]
  Row 39: [(13, 'Marq.Ass.')]
  Row 40: [(13, 'R.Salle'), (16, 'GILLET YVAN'), (26, 'ARA'), (30, '1488243')]
  Row 41: [(26, 'Juges\nLignes')]
  Row 43: [(26, 'Capitaines')]

=== PLAYERS TABLE ===
Players table: 4 rows x 6 cols
  Row 0: [(0, 'GRENOBLE V.UNIVERSITE CLUB'), (3, 'GPSO ACBB')]
  Row 1: [(0, 'N°'), (1, 'Nom Prénom'), (2, 'Licence'), (3, 'N°'), (4, 'Nom Pré

In [1]:
import sys
sys.path.insert(0, '/home/vincheetah/Documents/Programmation/Python/PyVolley/src')
from pathlib import Path
from pyvolley.parsers.v5 import MatchSheetParserV5

pdf_path = Path('/home/vincheetah/Documents/Programmation/Python/PyVolley/data/pdfs/2025-2026/ABCCS/Elite M. Poule A/ABCCS_EMA001.pdf')
v5 = MatchSheetParserV5()
r = v5.parse(pdf_path)
m = r.match

print("=== V5 PARSER — DETAILED OUTPUT ===")
print(f"Code: {m.code_match} | {m.competition} | {m.organisation}")
print(f"Lieu: {m.lieu} | Salle: {m.salle}")
print(f"Date: {m.date} {m.heure} | Score: {m.score_final} | Vainqueur: {m.vainqueur_nom}")
print(f"Durée: {m.duree_totale}")

print(f"\n--- Eq A: {m.equipe_a.nom} ({len(m.equipe_a.joueurs)} joueurs) ---")
print(f"  Officiels: {[(o.role, o.nom, o.prenom, o.licence) for o in m.equipe_a.officiels]}")
print(f"\n--- Eq B: {m.equipe_b.nom} ({len(m.equipe_b.joueurs)} joueurs) ---")
print(f"  Officiels: {[(o.role, o.nom, o.prenom, o.licence) for o in m.equipe_b.officiels]}")

print(f"\n--- Arbitres ({len(m.arbitres)}) ---")
for a in m.arbitres:
    print(f"  {a.role}: {a.nom} {a.prenom} [lic:{a.licence}] [ligue:{a.ligue}]")

print(f"\n--- Sets ({len(m.sets)}) ---")
for s in m.sets:
    print(f"\n  Set {s.numero}: {s.score_a}-{s.score_b} ({s.duree_minutes}min)")
    print(f"    Formation A: {s.formation_a.as_list() if s.formation_a else None}")
    print(f"    Formation B: {s.formation_b.as_list() if s.formation_b else None}")
    print(f"    TO A ({len(s.timeouts_a)}): {[(t.score_a, t.score_b) for t in s.timeouts_a]}")
    print(f"    TO B ({len(s.timeouts_b)}): {[(t.score_a, t.score_b) for t in s.timeouts_b]}")
    print(f"    Chg A ({len(s.changements_a)}):")
    for c in s.changements_a:
        print(f"      {c.joueur_entrant}<->{c.joueur_sortant} pos:{c.position} ({c.score_a}-{c.score_b})")
    print(f"    Chg B ({len(s.changements_b)}):")
    for c in s.changements_b:
        print(f"      {c.joueur_entrant}<->{c.joueur_sortant} pos:{c.position} ({c.score_a}-{c.score_b})")

chg_total = sum(len(s.changements_a) + len(s.changements_b) for s in m.sets)
to_total = sum(len(s.timeouts_a) + len(s.timeouts_b) for s in m.sets)
print(f"\n=== TOTALS: {chg_total} changements, {to_total} timeouts ===")

=== V5 PARSER — DETAILED OUTPUT ===
Code: EMA001 | EMA - ELITE MASCULINE - POULE A | Compétitions Nationales
Lieu: SAINT MARTIN D'HÈRES | Salle: CSU - GRAND GYMNASE
Date: 2025-09-20 20:30:00 | Score: 3/1 | Vainqueur: GRENOBLE V.UNIVERSITE
Durée: 2h9

--- Eq A: GRENOBLE V.UNIVERSITE CLUB (9 joueurs) ---
  Officiels: [('EA', 'VAN DEN ESHOF', 'DAMIEN', '1633339')]

--- Eq B: GPSO ACBB (11 joueurs) ---
  Officiels: [('EA', 'TANGUY', 'ERWAN', '1123609')]

--- Arbitres (4) ---
  RoleArbitre.PREMIER: CHOINARD MICHAEL [lic:1458903] [ligue:PAC]
  RoleArbitre.SECOND: HUMBERT RIDET DYLAN [lic:1762370] [ligue:PAC]
  RoleArbitre.MARQUEUR: AZAR NICOLAS [lic:2682444] [ligue:ARA]
  RoleArbitre.RESPONSABLE_SALLE: GILLET YVAN [lic:1488243] [ligue:ARA]

--- Sets (4) ---

  Set 1: 25-12 (24min)
    Formation A: ['51', '8', '11', '5', '17', '9']
    Formation B: ['3', '10', '24', '8', '7', '17']
    TO A (0): []
    TO B (2): [(5, 10), (6, 15)]
    Chg A (1):
      6<->9 pos:6 (24-12)
    Chg B (6):
      